In [0]:
import os, sys

REPO_ROOT = "/Workspace/Users/gakira.t1@gmail.com/Data-Health-FTW"
os.chdir(REPO_ROOT)
sys.path.insert(0, f"{REPO_ROOT}/src")
sys.dont_write_bytecode = True

from data_health_ftw.bronze import ingest as bronze_ingest
from data_health_ftw.silver import ingest as silver_ingest, validate as silver_validate
from data_health_ftw.gold import ingest as gold_ingest, validate as gold_validate

bronze_ingest.main()
silver_ingest.main()
silver_validate.main()
gold_ingest.main()
gold_validate.main()

In [0]:
%sql
SELECT version, COUNT(*) AS `rows`, COUNT(DISTINCT country_code) AS countries,
       ROUND(100 * AVG(CASE WHEN pm25_exposure IS NULL THEN 1 ELSE 0 END), 1) AS pm25_null_pct,
       ROUND(100 * AVG(CASE WHEN obesity_prev IS NULL THEN 1 ELSE 0 END), 1) AS obesity_null_pct,
       ROUND(100 * AVG(CASE WHEN smoking_prev IS NULL THEN 1 ELSE 0 END), 1) AS smoking_null_pct,
       ROUND(100 * AVG(CASE WHEN safe_water_pct IS NULL THEN 1 ELSE 0 END), 1) AS water_null_pct,
       ROUND(100 * AVG(CASE WHEN safe_sanitation_pct IS NULL THEN 1 ELSE 0 END), 1) AS sanitation_null_pct,
       ROUND(100 * AVG(CASE WHEN anaemia_prev_women IS NULL THEN 1 ELSE 0 END), 1) AS anaemia_null_pct
FROM (
  SELECT 'old' AS version, * FROM workspace.gold.gho_wide_backup
  UNION ALL
  SELECT 'new', * FROM workspace.gold.gho_wide
)
WHERE year BETWEEN 2010 AND 2023
GROUP BY version;

In [0]:
%sql
SELECT version, COUNT(*) AS `rows`, COUNT(DISTINCT country_code) AS countries,
       ROUND(100 * AVG(CASE WHEN pm25_exposure IS NULL THEN 1 ELSE 0 END), 1) AS pm25_null_pct,
       ROUND(100 * AVG(CASE WHEN obesity_prev IS NULL THEN 1 ELSE 0 END), 1) AS obesity_null_pct,
       ROUND(100 * AVG(CASE WHEN smoking_prev IS NULL THEN 1 ELSE 0 END), 1) AS smoking_null_pct,
       ROUND(100 * AVG(CASE WHEN safe_water_pct IS NULL THEN 1 ELSE 0 END), 1) AS water_null_pct,
       ROUND(100 * AVG(CASE WHEN safe_sanitation_pct IS NULL THEN 1 ELSE 0 END), 1) AS sanitation_null_pct,
       ROUND(100 * AVG(CASE WHEN anaemia_prev_women IS NULL THEN 1 ELSE 0 END), 1) AS anaemia_null_pct
FROM (
  SELECT 'old' AS version, country_code, year, pm25_exposure, obesity_prev, smoking_prev,
         safe_water_pct, safe_sanitation_pct, anaemia_prev_women
  FROM workspace.gold.gho_wide_backup
  UNION ALL
  SELECT 'new', country_code, year, pm25_exposure, obesity_prev, smoking_prev,
         safe_water_pct, safe_sanitation_pct, anaemia_prev_women
  FROM workspace.gold.gho_wide
)
WHERE year BETWEEN 2010 AND 2023
GROUP BY version;

In [0]:
import requests, pandas as pd

BASE = "https://xmart-api-public.who.int/DATA_/RELAY_PSCP"
SELECT = ("IND_CODE,DIM_TIME,DIM_PUBLISH_STATE_CODE,DIM_1_CODE,DIM_MEMBER_1_CODE,"
          "DIM_2_CODE,DIM_MEMBER_2_CODE")
for code in ["GHED_CHE_PC_USD", "GHED_CHE_NCU", "GHED_GGHED_NCU", "GHED_GGHED_GDP"]:
    rows, skip = [], 0
    while True:
        batch = requests.get(BASE, params={
            "$filter": f"IND_CODE eq '{code}' and DIM_GEO_CODE_TYPE eq 'COUNTRY'",
            "$select": SELECT, "$top": 5000, "$skip": skip}, timeout=120).json().get("value", [])
        if not batch:
            break
        rows += batch
        skip += len(batch)
    df = pd.DataFrame(rows).fillna("-")
    print(code, len(df))
    if len(df):
        print(df.groupby([c for c in df.columns if c != "DIM_TIME"])
                .agg(n=("DIM_TIME", "size"), y0=("DIM_TIME", "min"), y1=("DIM_TIME", "max"))
                .to_string(), "\n")

In [0]:
import os, sys
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

REPO_ROOT = "/Workspace/Users/gakira.t1@gmail.com/Data-Health-FTW"
os.chdir(REPO_ROOT)
sys.path.insert(0, f"{REPO_ROOT}/src")

import pytest
pytest.main([
    "tests/test_config.py",
    "tests/test_gho_client.py",
    "tests/test_bronze_transform.py",
    "tests/test_gold_transform.py",
    "-q", "-p", "no:cacheprovider",
])